In [1]:
!pip install causal-conv1d>=1.2.0
!pip install mamba-ssm
!pip install jsonlines

  error: subprocess-exited-with-error
  
  × Building wheel for causal-conv1d (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for causal-conv1d
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (causal-conv1d)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 3.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for mamba-ssm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for mamba-ssm
Failed to build mamba-ssm
ERROR: ERROR: Failed to build installable wheels for

In [11]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class THUMOS14_I3D_Dataset(Dataset):
    def __init__(self, data_root, split='train', fps=25.0, clip_length=16):
        self.data_root = data_root
        self.split = split
        self.fps = fps
        self.clip_length = clip_length
        
        self.rgb_dir = os.path.join(data_root, 'features', split, 'rgb')
        self.flow_dir = os.path.join(data_root, 'features', split, 'flow')
        
        split_file = os.path.join(data_root, f'split_{split}.txt')
        with open(split_file, 'r') as f:
            self.video_ids = [line.strip() for line in f.readlines()]
            
        with open(os.path.join(data_root, 'gt.json'), 'r') as f:
            self.ground_truth = json.load(f)

        # --- THE RAM CACHE ---
        self.data_cache = []
        print(f"Loading {split} dataset into RAM. This will take about 60 seconds...")
        
        for vid_id in tqdm(self.video_ids, desc=f"Caching {split} Features"):
            try:
                rgb_feat = np.load(os.path.join(self.rgb_dir, f'{vid_id}.npy'))
                flow_feat = np.load(os.path.join(self.flow_dir, f'{vid_id}.npy'))
                fused_features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32)
            except FileNotFoundError:
                continue # Skip missing videos silently during caching
                
            num_vectors = fused_features.shape[0] 
            target_mask = torch.zeros(num_vectors, dtype=torch.float32)
            
            if vid_id in self.ground_truth['database']:
                annotations = self.ground_truth['database'][vid_id]['annotations']
                for ann in annotations:
                    start_sec = float(ann['segment'][0])
                    end_sec = float(ann['segment'][1])
                    
                    start_idx = int((start_sec * self.fps) / self.clip_length)
                    end_idx = int((end_sec * self.fps) / self.clip_length)
                    
                    start_idx = max(0, min(start_idx, num_vectors - 1))
                    end_idx = max(0, min(end_idx, num_vectors - 1))
                    
                    if start_idx <= end_idx:
                        target_mask[start_idx:end_idx + 1] = 1.0
            
            # Store the fully processed tensors in memory
            self.data_cache.append((fused_features, target_mask, vid_id))

    def __len__(self):
        return len(self.data_cache)

    def __getitem__(self, idx):
        # Instantly return the pre-loaded tensors from RAM
        return self.data_cache[idx]

In [ ]:
import os 
os.cpu_count()

In [12]:
# 1. Update the DATA_ROOT to include the nested THUMOS14 folder
DATA_ROOT = '/kaggle/input/datasets/valuejack/thumos14-asl/THUMOS14' 

# 2. Run the Dataloader again
train_dataset = THUMOS14_I3D_Dataset(DATA_ROOT, split='train')

train_loader = DataLoader(
    train_dataset, 
    batch_size=1, 
    shuffle=True,
    num_workers=0, # Reset to 0
    pin_memory=True 
)

# 3. Fetch just one video to verify
features, mask, vid_id = next(iter(train_loader))

print(f"Video ID: {vid_id[0]}")
print(f"Feature Tensor Shape: {features.shape} -> (Batch, Sequence_Length, Features)")
print(f"Mask Tensor Shape: {mask.shape} -> (Batch, Sequence_Length)")
print(f"Total Action Vectors in this video: {int(mask.sum().item())} out of {mask.shape[1]}")

Loading train dataset into RAM. This will take about 60 seconds...


Caching train Features:   0%|          | 0/200 [00:00<?, ?it/s]

Video ID: video_validation_0000182
Feature Tensor Shape: torch.Size([1, 158, 2048]) -> (Batch, Sequence_Length, Features)
Mask Tensor Shape: torch.Size([1, 158]) -> (Batch, Sequence_Length)
Total Action Vectors in this video: 63 out of 158


In [ ]:
next(iter(train_loader))

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        """
        alpha: Balances the importance of positive/negative examples.
               Since actions are rare, alpha gives them more weight.
        gamma: The focusing parameter. Higher values penalize the model 
               more for missing the rare action frames.
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs are raw logits before sigmoid
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss) # Probability of the correct class
        
        # Calculate Focal Loss
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

In [14]:
import torch
import torch.nn as nn
from transformers import MambaConfig, MambaModel

class MambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, d_state=16, d_conv=4, expand=2):
        super(MambaGlobalScanner, self).__init__()
        
        # 1. Dimensionality Reduction
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # 2. The Core Mamba Block (Using Hugging Face Native Implementation)
        config = MambaConfig(
            hidden_size=d_model,  # Maps to d_model
            state_size=d_state,   # Maps to d_state
            conv_kernel=d_conv,   # Maps to d_conv
            expand=expand         # Maps to expand factor
        )
        # We use MambaModel natively without a pre-trained head
        self.mamba = MambaModel(config)
        
        # 3. The Classification Head
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x):
        # x shape: (Batch, Sequence_Length, 2048)
        
        # Project inputs
        x = self.input_proj(x)  # Shape: (Batch, Sequence, 256)
        
        # Global Temporal Scanning via Mamba
        # HF Mamba takes inputs_embeds and returns a model output object
        outputs = self.mamba(inputs_embeds=x)
        
        # Extract the hidden states processed by the SSM
        hidden_states = outputs.last_hidden_state  # Shape: (Batch, Sequence, 256)
        
        # Classify each timestep
        logits = self.classifier(hidden_states) # Shape: (Batch, Sequence, 1)
        
        # Remove the last dimension so it matches our target mask shape
        return logits.squeeze(-1)

In [15]:
# Instantiate the model
model = MambaGlobalScanner()

# Run our single batch from the Dataloader Sanity Check through the model
predicted_logits = model(features)

print(f"Input Feature Shape: {features.shape}")
print(f"Output Logits Shape: {predicted_logits.shape}")
print(f"Target Mask Shape:   {mask.shape}")
print("\nSuccess! The shapes match perfectly.")

Input Feature Shape: torch.Size([1, 158, 2048])
Output Logits Shape: torch.Size([1, 158])
Target Mask Shape:   torch.Size([1, 158])

Success! The shapes match perfectly.


In [16]:
import torch.optim as optim
from tqdm.auto import tqdm

# Hyperparameters
EPOCHS = 10
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training on: {DEVICE}")

# 1. Initialize Model, Loss, and Optimizer
# We instantiate fresh to ensure clean weights
mamba_model = MambaGlobalScanner().to(DEVICE)

# Alpha=0.85 gives heavy weight to the rare action frames (1s) over background (0s)
criterion = FocalLoss(alpha=0.85, gamma=2.0).to(DEVICE) 
optimizer = optim.AdamW(mamba_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_recall = 0
    valid_batches = 0 # To count batches that actually contain actions

    # Progress bar for visual tracking
    pbar = tqdm(dataloader, desc="Training Epoch")

    for features, targets, _ in pbar:
        # Move tensors to GPU
        features, targets = features.to(device), targets.to(device)

        # Forward Pass
        optimizer.zero_grad()
        logits = model(features)
        
        # Calculate Focal Loss
        loss = criterion(logits, targets)
        
        # Backward Pass & Optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate Recall (True Positives / Actual Positives)
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float() # Threshold at 0.5

            actual_positives = targets.sum().item()
            if actual_positives > 0:
                true_positives = (preds * targets).sum().item()
                recall = true_positives / actual_positives
                total_recall += recall
                valid_batches += 1
            else:
                recall = 0.0 # No actions in this specific video

        pbar.set_postfix(Loss=f"{loss.item():.4f}", Recall=f"{recall:.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_recall = total_recall / valid_batches if valid_batches > 0 else 0
    
    return avg_loss, avg_recall

# 2. Run the Training Loop
for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    avg_loss, avg_recall = train_epoch(mamba_model, train_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Avg Recall: {avg_recall:.4f}")

# Save the trained scanner!
torch.save(mamba_model.state_dict(), "mamba_scanner_stage1.pth")
print("\nModel saved successfully!")

Training on: cuda

--- Epoch 1/10 ---


Training Epoch:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 1 Complete | Avg Loss: 0.1288 | Avg Recall: 0.3576

--- Epoch 2/10 ---


Training Epoch:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 2 Complete | Avg Loss: 0.1045 | Avg Recall: 0.5808

--- Epoch 3/10 ---


...

Epoch 8 Complete | Avg Loss: 0.0753 | Avg Recall: 0.6780

--- Epoch 9/10 ---


Training Epoch:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 9 Complete | Avg Loss: 0.0755 | Avg Recall: 0.6895

--- Epoch 10/10 ---


Training Epoch:   0%|          | 0/200 [00:00<?, ?it/s]

Epoch 10 Complete | Avg Loss: 0.0695 | Avg Recall: 0.7139

Model saved successfully!


In [18]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class TransformerRefinerDataset(Dataset):
    def __init__(self, data_root, split='train', window_size=64, fps=25.0, clip_length=16):
        self.window_size = window_size
        self.fps = fps
        self.clip_length = clip_length
        
        self.rgb_dir = os.path.join(data_root, 'features', split, 'rgb')
        self.flow_dir = os.path.join(data_root, 'features', split, 'flow')
        
        with open(os.path.join(data_root, f'split_{split}.txt'), 'r') as f:
            video_ids = [line.strip() for line in f.readlines()]
            
        with open(os.path.join(data_root, 'gt.json'), 'r') as f:
            ground_truth = json.load(f)

        # 1. Load the full videos into a RAM Cache dictionary
        self.video_cache = {}
        print(f"Caching {split} features into RAM...")
        for vid_id in tqdm(video_ids, desc="Loading Features"):
            try:
                rgb_feat = np.load(os.path.join(self.rgb_dir, f'{vid_id}.npy'))
                flow_feat = np.load(os.path.join(self.flow_dir, f'{vid_id}.npy'))
                self.video_cache[vid_id] = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32)
            except FileNotFoundError:
                continue

        # 2. The Teacher Forcing Logic: Extract just the action snippets
        self.snippets = []
        for vid_id in self.video_cache.keys():
            if vid_id in ground_truth['database']:
                for ann in ground_truth['database'][vid_id]['annotations']:
                    start_sec = float(ann['segment'][0])
                    end_sec = float(ann['segment'][1])
                    
                    # Convert to vector indices
                    s_idx = int((start_sec * self.fps) / self.clip_length)
                    e_idx = int((end_sec * self.fps) / self.clip_length)
                    
                    # Center a fixed-size window around the action
                    center_idx = (s_idx + e_idx) // 2
                    w_start = center_idx - (self.window_size // 2)
                    w_end = w_start + self.window_size
                    
                    # Calculate targets relative strictly to the new 64-frame window
                    local_start = s_idx - w_start
                    local_end = e_idx - w_start
                    
                    # Normalize boundaries between 0.0 and 1.0 for the Transformer
                    t_start = max(0.0, min(1.0, local_start / self.window_size))
                    t_end = max(0.0, min(1.0, local_end / self.window_size))
                    
                    self.snippets.append((vid_id, w_start, w_end, t_start, t_end))

    def __len__(self):
        return len(self.snippets)

    def __getitem__(self, idx):
        vid_id, w_start, w_end, t_start, t_end = self.snippets[idx]
        full_video = self.video_cache[vid_id]
        num_vecs = full_video.shape[0]
        
        # Initialize a blank snippet of exactly size 64x2048 (padded with zeros)
        snippet = torch.zeros((self.window_size, full_video.shape[1]), dtype=torch.float32)
        
        # Calculate safe bounds (in case the window extends past the start/end of the video)
        safe_start = max(0, w_start)
        safe_end = min(num_vecs, w_end)
        
        # Calculate where to insert the data into our blank snippet
        insert_start = safe_start - w_start
        insert_end = insert_start + (safe_end - safe_start)
        
        if safe_end > safe_start:
            snippet[insert_start:insert_end] = full_video[safe_start:safe_end]
            
        targets = torch.tensor([t_start, t_end], dtype=torch.float32)
        
        return snippet, targets

In [19]:
# Assuming DATA_ROOT is still pointing to your THUMOS14 folder
refiner_dataset = TransformerRefinerDataset(DATA_ROOT, split='train', window_size=64)

# We can use a larger batch size now because the sequences are only 64 frames long!
refiner_loader = DataLoader(refiner_dataset, batch_size=32, shuffle=True)

# Fetch one batch
snippets, targets = next(iter(refiner_loader))

print(f"\nTotal Action Snippets Extracted: {len(refiner_dataset)}")
print(f"Batch Snippet Shape: {snippets.shape} -> (Batch, Sequence_Length, Features)")
print(f"Batch Targets Shape: {targets.shape} -> (Batch, [Start_Offset, End_Offset])")

# Look at the first snippet's target values
print(f"Sample Normalized Target: Start {targets[0][0]:.4f}, End {targets[0][1]:.4f}")

Caching train features into RAM...


Loading Features:   0%|          | 0/200 [00:00<?, ?it/s]


Total Action Snippets Extracted: 3007
Batch Snippet Shape: torch.Size([32, 64, 2048]) -> (Batch, Sequence_Length, Features)
Batch Targets Shape: torch.Size([32, 2]) -> (Batch, [Start_Offset, End_Offset])
Sample Normalized Target: Start 0.4219, End 0.5938


In [21]:
import torch.nn as nn
import math

class TransformerRefiner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, nhead=4, num_layers=1, window_size=64):
        super(TransformerRefiner, self).__init__()
        
        # 1. Dimensionality Reduction (Matches the Mamba scanner's dimension)
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # 2. Positional Encoding (Crucial for Transformers to understand time)
        # We use a simple learnable parameter since our window size is fixed at 64
        self.pos_encoder = nn.Parameter(torch.zeros(1, window_size, d_model))
        
        # 3. The Dense Attention Block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, 
            dropout=0.1, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. The Boundary Regressor Head
        # Predicts exactly 2 continuous numbers (Start Offset, End Offset)
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid() # Forces the output strictly between 0.0 and 1.0
        )

    def forward(self, x):
        # x shape: (Batch, 64, 2048)
        
        # Project and add positional context
        x = self.input_proj(x) + self.pos_encoder # Shape: (Batch, 64, 256)
        
        # Apply dense self-attention
        x = self.transformer(x) # Shape: (Batch, 64, 256)
        
        # Temporal Pooling: Collapse the 64 frames into a single contextual vector
        # We take the mean across the sequence dimension (dim=1)
        x_pooled = x.mean(dim=1) # Shape: (Batch, 256)
        
        # Predict boundaries
        boundaries = self.regressor(x_pooled) # Shape: (Batch, 2)
        
        return boundaries

In [22]:
import torch.optim as optim
from tqdm.auto import tqdm

# Hyperparameters
EPOCHS = 15
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training Refiner on: {DEVICE}")

# Initialize Model, Loss, and Optimizer
refiner_model = TransformerRefiner().to(DEVICE)
criterion = nn.SmoothL1Loss().to(DEVICE) 
optimizer = optim.AdamW(refiner_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def train_refiner_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_mae = 0 # Mean Absolute Error for human-readable tracking

    pbar = tqdm(dataloader, desc="Training Refiner")

    for snippets, targets in pbar:
        snippets, targets = snippets.to(device), targets.to(device)

        # Forward Pass
        optimizer.zero_grad()
        predictions = model(snippets)
        
        # Calculate Smooth L1 Loss
        loss = criterion(predictions, targets)
        
        # Backward Pass & Optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate pure L1 distance (MAE) just for our progress bar
        with torch.no_grad():
            mae = torch.abs(predictions - targets).mean().item()
            total_mae += mae

        pbar.set_postfix(Loss=f"{loss.item():.4f}", MAE=f"{mae:.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_mae = total_mae / len(dataloader)
    
    return avg_loss, avg_mae

# Run the Training Loop
for epoch in range(1, EPOCHS + 1):
    avg_loss, avg_mae = train_refiner_epoch(refiner_model, refiner_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Avg MAE: {avg_mae:.4f}")

# Save the trained refiner!
torch.save(refiner_model.state_dict(), "transformer_refiner_stage2.pth")
print("\nRefiner Model saved successfully!")

Training Refiner on: cuda


Training Refiner:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 1 Complete | Avg Loss: 0.0009 | Avg MAE: 0.0262


Training Refiner:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 2 Complete | Avg Loss: 0.0006 | Avg MAE: 0.0225


...

Epoch 13 Complete | Avg Loss: 0.0003 | Avg MAE: 0.0166


Training Refiner:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 14 Complete | Avg Loss: 0.0003 | Avg MAE: 0.0169


Training Refiner:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 15 Complete | Avg Loss: 0.0003 | Avg MAE: 0.0157

Refiner Model saved successfully!


In [23]:
import os
import json
import numpy as np
import torch
from tqdm.auto import tqdm

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = '/kaggle/input/datasets/valuejack/thumos14-asl/THUMOS14' # Update if needed
SPLIT = 'test'
FPS = 25.0
CLIP_LENGTH = 16
WINDOW_SIZE = 64

print("Loading Models for Inference...")

# 1. Load the Trained Models
mamba_model = MambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load("mamba_scanner_stage1.pth", weights_only=True))
mamba_model.eval()

refiner_model = TransformerRefiner().to(DEVICE)
refiner_model.load_state_dict(torch.load("transformer_refiner_stage2.pth", weights_only=True))
refiner_model.eval()

# 2. Setup Test Data Paths
rgb_dir = os.path.join(DATA_ROOT, 'features', SPLIT, 'rgb')
flow_dir = os.path.join(DATA_ROOT, 'features', SPLIT, 'flow')

with open(os.path.join(DATA_ROOT, f'split_{SPLIT}.txt'), 'r') as f:
    test_videos = [line.strip() for line in f.readlines()]

final_predictions = {}

# 3. The Inference Loop
with torch.no_grad():
    for vid_id in tqdm(test_videos, desc="Evaluating Test Set"):
        try:
            # Load Full Video Features
            rgb_feat = np.load(os.path.join(rgb_dir, f'{vid_id}.npy'))
            flow_feat = np.load(os.path.join(flow_dir, f'{vid_id}.npy'))
            video_features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        except FileNotFoundError:
            continue
            
        num_vecs = video_features.shape[1]
        if num_vecs == 0: continue

        final_predictions[vid_id] = []

        # --- STAGE I: Global Scan ---
        mamba_logits = mamba_model(video_features)
        mamba_probs = torch.sigmoid(mamba_logits).squeeze(0).cpu().numpy()
        
        # Find candidate segments (probability > 0.5)
        # We look for continuous blocks of 1s in a boolean array
        is_action = mamba_probs > 0.5
        
        # A clever way to find start/end indices of contiguous blocks
        padded = np.pad(is_action, (1, 1), 'constant')
        diffs = np.diff(padded.astype(int))
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0] - 1 

        # --- STAGE II: Local Refinement ---
        for s_idx, e_idx in zip(starts, ends):
            # 1. Determine the 64-frame window center
            center_idx = (s_idx + e_idx) // 2
            w_start = center_idx - (WINDOW_SIZE // 2)
            w_end = w_start + WINDOW_SIZE
            
            # 2. Extract and Pad the Snippet
            snippet = torch.zeros((1, WINDOW_SIZE, 2048), dtype=torch.float32).to(DEVICE)
            
            safe_start = max(0, w_start)
            safe_end = min(num_vecs, w_end)
            
            insert_start = safe_start - w_start
            insert_end = insert_start + (safe_end - safe_start)
            
            if safe_end > safe_start:
                snippet[0, insert_start:insert_end] = video_features[0, safe_start:safe_end]
                
            # 3. Predict Normalized Boundaries
            pred_bounds = refiner_model(snippet).squeeze(0).cpu().numpy()
            t_start, t_end = pred_bounds[0], pred_bounds[1]
            
            # 4. Reverse Mapping: Normalized (0-1) -> Absolute Indices
            abs_start_idx = w_start + (t_start * WINDOW_SIZE)
            abs_end_idx = w_start + (t_end * WINDOW_SIZE)
            
            # Ensure boundaries are logical
            abs_start_idx = max(0, min(num_vecs, abs_start_idx))
            abs_end_idx = max(0, min(num_vecs, abs_end_idx))
            
            if abs_start_idx >= abs_end_idx:
                continue # Model predicted an invalid or collapsed window
                
            # 5. Final Conversion: Indices -> Absolute Seconds
            start_sec = (abs_start_idx * CLIP_LENGTH) / FPS
            end_sec = (abs_end_idx * CLIP_LENGTH) / FPS
            
            # Use Mamba's peak probability inside this window as the confidence score
            confidence = float(np.max(mamba_probs[s_idx:e_idx+1]))
            
            final_predictions[vid_id].append({
                "segment": [float(start_sec), float(end_sec)],
                "score": confidence
            })

# Save results to JSON for mAP evaluation
with open("submission_predictions.json", "w") as f:
    json.dump({"results": final_predictions}, f, indent=4)

print("\nInference Complete! Saved to submission_predictions.json")

Loading Models for Inference...


Evaluating Test Set:   0%|          | 0/210 [00:00<?, ?it/s]


Inference Complete! Saved to submission_predictions.json


In [25]:
import json
import numpy as np

# --- Configuration ---
GT_FILE = '/kaggle/input/datasets/valuejack/thumos14-asl/THUMOS14/gt.json' # Update to your exact gt.json path
PRED_FILE = 'submission_predictions.json'
IOU_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

def compute_1d_iou(segment1, segment2):
    """Calculates Intersection over Union for two 1D temporal segments."""
    intersection_start = max(segment1[0], segment2[0])
    intersection_end = min(segment1[1], segment2[1])
    intersection = max(0, intersection_end - intersection_start)
    
    union = (segment1[1] - segment1[0]) + (segment2[1] - segment2[0]) - intersection
    if union <= 0: return 0.0
    return intersection / union

# 1. Load Data
with open(GT_FILE, 'r') as f:
    ground_truth = json.load(f)['database']
    
with open(PRED_FILE, 'r') as f:
    predictions = json.load(f)['results']

print("Calculating Average Precision (AP) at various IoU thresholds...\n")

# 2. Evaluation Loop
ap_scores = []

for iou_thresh in IOU_THRESHOLDS:
    all_scores = []
    all_matches = [] # 1 for True Positive, 0 for False Positive
    total_gt_actions = 0
    
    for vid_id, preds in predictions.items():
        if vid_id not in ground_truth: continue
            
        gt_annos = ground_truth[vid_id]['annotations']
        
        # THE FIX: Cast the JSON strings to floats
        gt_segments = [[float(ann['segment'][0]), float(ann['segment'][1])] for ann in gt_annos]
        total_gt_actions += len(gt_segments)
        
        # Sort predictions by confidence score (descending)
        preds = sorted(preds, key=lambda x: x['score'], reverse=True)
        
        gt_matched = [False] * len(gt_segments)
        
        for p in preds:
            all_scores.append(p['score'])
            
            best_iou = 0
            best_gt_idx = -1
            
            # Find the best matching ground truth segment
            for i, gt_seg in enumerate(gt_segments):
                if gt_matched[i]: continue
                iou = compute_1d_iou(p['segment'], gt_seg)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = i
                    
            if best_iou >= iou_thresh:
                all_matches.append(1)
                gt_matched[best_gt_idx] = True # Mark this GT as found
            else:
                all_matches.append(0)

    # 3. Calculate Precision-Recall Curve
    # Sort all predictions globally by confidence score
    sorted_indices = np.argsort(all_scores)[::-1]
    all_matches = np.array(all_matches)[sorted_indices]
    
    cumulative_tp = np.cumsum(all_matches)
    cumulative_fp = np.cumsum(1 - all_matches)
    
    precision = cumulative_tp / (cumulative_tp + cumulative_fp + 1e-8)
    recall = cumulative_tp / (total_gt_actions + 1e-8)
    
    # 4. Calculate Average Precision (Area under the PR curve)
    ap = 0.0
    # 11-point interpolation mapping
    for t in np.arange(0, 1.1, 0.1):
        if np.sum(recall >= t) == 0:
            p = 0
        else:
            p = np.max(precision[recall >= t])
        ap += p / 11.0
        
    ap_scores.append(ap)
    print(f"AP @ IoU {iou_thresh:.1f} : {ap * 100:.2f}%")

print("-" * 30)
print(f"Average mAP : {np.mean(ap_scores) * 100:.2f}%")

Calculating Average Precision (AP) at various IoU thresholds...

AP @ IoU 0.3 : 64.63%
AP @ IoU 0.4 : 53.35%
AP @ IoU 0.5 : 35.75%
AP @ IoU 0.6 : 24.93%
AP @ IoU 0.7 : 15.85%
------------------------------
Average mAP : 38.90%


In [ ]:
# 1. Clone the gold-standard extraction repository
!git clone https://github.com/v-iashin/video_features.git
%cd video_features

# 2. Install the specific dependencies
!pip install omegaconf
!pip install timm
!pip install moviepy

In [2]:
!find /kaggle/input -name "*istockphoto*.npy"

In [4]:
import torch
import torch.nn as nn
import numpy as np
import os

# --- 1. MODEL ARCHITECTURES (Must be defined to load weights) ---

class MambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512):
        super().__init__()
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        x = self.proj(x)
        x, _ = self.rnn(x)
        return self.classifier(x)

class TransformerRefiner(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512, nhead=8):
        super().__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.encoder_layer, num_layers=1)
        self.fc = nn.Linear(input_dim, 2)
    def forward(self, x):
        x = self.transformer(x)
        x = torch.mean(x, dim=1)
        return torch.sigmoid(self.fc(x))

# --- 2. SETUP & LOADING ---

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FPS = 23.97
CLIP_LENGTH = 16 
WINDOW_SIZE = 64

# REPLACE THESE with the exact output from your !find command
rgb_path = "PASTE_YOUR_RGB_PATH_HERE.npy"
flow_path = "PASTE_YOUR_FLOW_PATH_HERE.npy"

print("Initializing Models and Loading Weights...")
mamba_model = MambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load("mamba_scanner_stage1.pth", map_location=DEVICE))

refiner_model = TransformerRefiner().to(DEVICE)
refiner_model.load_state_dict(torch.load("transformer_refiner_stage2.pth", map_location=DEVICE))

mamba_model.eval()
refiner_model.eval()

# --- 3. FEATURE PROCESSING ---

print("Loading Extracted Features...")
rgb_feat = np.load(rgb_path)
flow_feat = np.load(flow_path)
fused_features = np.concatenate((rgb_feat, flow_feat), axis=-1)
video_tensor = torch.tensor(fused_features, dtype=torch.float32).unsqueeze(0).to(DEVICE)
num_vecs = video_tensor.shape[1]

# --- 4. INFERENCE ---

with torch.no_grad():
    mamba_logits = mamba_model(video_tensor)
    mamba_probs = torch.sigmoid(mamba_logits).squeeze(0).cpu().numpy()
    
    is_action = mamba_probs > 0.5
    padded = np.pad(is_action, (1, 1), 'constant')
    diffs = np.diff(padded.astype(int))
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0] - 1 

    final_predictions = []

    for s_idx, e_idx in zip(starts, ends):
        center_idx = (s_idx + e_idx) // 2
        w_start = center_idx - (WINDOW_SIZE // 2)
        w_end = w_start + WINDOW_SIZE
        
        snippet = torch.zeros((1, WINDOW_SIZE, 2048), dtype=torch.float32).to(DEVICE)
        safe_start, safe_end = max(0, w_start), min(num_vecs, w_end)
        insert_start = safe_start - w_start
        insert_end = insert_start + (safe_end - safe_start)
        
        if safe_end > safe_start:
            snippet[0, insert_start:insert_end] = video_tensor[0, safe_start:safe_end]
            
        pred_bounds = refiner_model(snippet).squeeze(0).cpu().numpy()
        abs_start_idx = max(0, min(num_vecs, w_start + (pred_bounds[0] * WINDOW_SIZE)))
        abs_end_idx = max(0, min(num_vecs, w_start + (pred_bounds[1] * WINDOW_SIZE)))
        
        if abs_start_idx < abs_end_idx:
            start_sec = (abs_start_idx * CLIP_LENGTH) / FPS
            end_sec = (abs_end_idx * CLIP_LENGTH) / FPS
            conf = float(np.max(mamba_probs[s_idx:e_idx+1]))
            final_predictions.append((start_sec, end_sec, conf))

print("\n" + "="*30)
print("FINAL DEMO PREDICTIONS")
print("="*30)
if not final_predictions:
    print("No action detected. (Try lowering the threshold from 0.5 if it's a subtle action)")
else:
    for i, (start, end, conf) in enumerate(final_predictions):
        print(f"Action {i+1}: {start:.2f}s to {end:.2f}s (Confidence: {conf*100:.1f}%)")

Initializing Models and Loading Weights...


FileNotFoundError: [Errno 2] No such file or directory: 'mamba_scanner_stage1.pth'

In [2]:
!find /kaggle/input -name "*istockphoto*.npy"

/kaggle/input/datasets/amancer/demo-video-features/istockphoto-1455749437-640_adpp_is_flow.npy
/kaggle/input/datasets/amancer/demo-video-features/istockphoto-1455749437-640_adpp_is_rgb.npy
